# NB1 — Polyvore1000 Data Processing Experiment

Notebook này chỉ dùng để **experiment / debug** pipeline tạo compatibility dataset.

Core logic nằm ở:

`src/data/build_compatibility_dataset.py`

Notebook không mount Google Drive và không chứa bản sao production logic. Mặc định notebook chỉ chạy một debug subset nhỏ để tránh vô tình regenerate toàn bộ benchmark.


## 1. Locate repository và import core pipeline

Chạy notebook từ bất kỳ thư mục con nào trong repo. Cell dưới sẽ tìm repo root dựa trên file `src/data/build_compatibility_dataset.py`.


In [ ]:
from pathlib import Path
import json
import sys


def find_repo_root(start: Path = Path.cwd()) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "src/data/build_compatibility_dataset.py").exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy src/data/build_compatibility_dataset.py. "
        "Hãy đặt notebook trong repo opisoverated hoặc đổi working directory về repo."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.build_compatibility_dataset import build_compatibility_dataset

print("Repo root:", REPO_ROOT)


## 2. Experiment config

`DEBUG_LIMIT = 20` là mặc định an toàn cho notebook. Output experiment được tách riêng khỏi artifact benchmark chính.


In [ ]:
DATASET_NAME = "codewaly/polyvore1000"
SPLIT = "train"
SEED = 42
NEGATIVES_PER_OUTFIT = 1
MIN_ITEMS_PER_KIT = 2
DEBUG_LIMIT = 20

OUTPUT_FILE = (
    REPO_ROOT
    / "data/processed/experiments"
    / f"polyvore1000_compatibility_{SPLIT}_debug.jsonl"
)

print("Experiment output:", OUTPUT_FILE)


## 3. Chạy debug experiment

Pipeline sẽ thực hiện: load dataset → build indexes → positive/negative generation → canonical Data Contract validation.


In [ ]:
stats = build_compatibility_dataset(
    output_file=OUTPUT_FILE,
    dataset_name=DATASET_NAME,
    split=SPLIT,
    seed=SEED,
    negatives_per_outfit=NEGATIVES_PER_OUTFIT,
    min_items_per_kit=MIN_ITEMS_PER_KIT,
    debug_limit=DEBUG_LIMIT,
)

stats


## 4. Inspect canonical JSONL samples

Positive phải có `negative_metadata = null`. Negative phải có `sample_id`, `source_kit_id` và provenance nằm trong `negative_metadata`.


In [ ]:
def read_jsonl_head(path: Path, n: int = 6):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for index, line in enumerate(f):
            if index >= n:
                break
            rows.append(json.loads(line))
    return rows


samples = read_jsonl_head(OUTPUT_FILE, n=6)
for sample in samples:
    print(json.dumps(sample, ensure_ascii=False, indent=2))
    print("-" * 80)


## 5. Kiểm tra nhanh positive/negative family

Cell này giúp xác nhận mỗi sample vẫn truy ngược được về `source_kit_id` và positive/negative dùng canonical IDs.


In [ ]:
for sample in samples:
    print(
        sample["sample_id"],
        "| source:", sample["source_kit_id"],
        "| label:", sample["label"],
        "| items:", len(sample["items"]),
    )


## 6. Full run — chỉ bật khi muốn tạo artifact chính

Không chạy cell này trong experiment thông thường. Dataset V1 đã freeze thì không nên regenerate âm thầm. Đổi `RUN_FULL = True` chỉ khi chủ động tạo/rebuild artifact theo protocol của team.


In [ ]:
RUN_FULL = False

if RUN_FULL:
    full_output = (
        REPO_ROOT
        / "data/processed"
        / f"polyvore1000_compatibility_{SPLIT}.jsonl"
    )

    full_stats = build_compatibility_dataset(
        output_file=full_output,
        dataset_name=DATASET_NAME,
        split=SPLIT,
        seed=SEED,
        negatives_per_outfit=NEGATIVES_PER_OUTFIT,
        min_items_per_kit=MIN_ITEMS_PER_KIT,
        debug_limit=None,
    )

    print(full_output)
    display(full_stats)
else:
    print("RUN_FULL=False — không tạo full benchmark artifact.")
